In [2]:
import requests
import time
import pandas as pd

url = "https://api.platform.opentargets.org/api/v4/graphql"

query = """
query RAdiseaseTargets($efoId: String!, $page: Pagination!) {
  disease(efoId: $efoId) {
    associatedTargets(page: $page) {
      count
      rows {
        target {
          id
          approvedSymbol
          approvedName
        }
        score
        datatypeScores {
          id
          score
        }
      }
    }
  }
}
"""

def fetch_page(variables, retries=3, delay=2):
    result = None
    for attempt in range(retries):
        response = requests.post(url, json={"query": query, "variables": variables})
        result = response.json()
        if result.get("data", {}).get("disease") is not None:
            return result
        print(f"Attempt {attempt+1} failed (disease=None), retrying in {delay}s...")
        time.sleep(delay)
        delay *= 2
    return result

all_genes = []
page_size = 50
page_index = 0

while True:
    variables = {
        "efoId": "EFO_0000685",
        "page": {"size": page_size, "index": page_index}
    }

    result = fetch_page(variables)

    if result.get("data", {}).get("disease") is None:
        print("Still failing after retries. Full response:")
        print(result)
        break

    rows = result["data"]["disease"]["associatedTargets"]["rows"]
    total = result["data"]["disease"]["associatedTargets"]["count"]

    if not rows:
        break

    all_genes.extend(rows)
    print(f"Fetched {len(all_genes)} / {total}")
    page_index += 1

    if len(all_genes) >= total:
        break

print(f"\nTotal genes fetched: {len(all_genes)}")

# Once it works, save it
df = pd.json_normalize(all_genes)
df.to_csv("opentargets_RA_genes.tsv", sep="\t", index=False)

Still failing after retries. Full response:
{'data': {'disease': None}}

Total genes fetched: 0


In [5]:
# Parse into clean DataFrame
records = []
for row in all_genes:
    record = {
        "geneSymbol": row["target"]["approvedSymbol"],
        "ensemblId": row["target"]["id"],
        "geneName": row["target"]["approvedName"],
        "overallScore": row["score"]
    }
    # Add individual datatype scores
    for dt in row["datatypeScores"]:
        record[dt["id"]] = dt["score"]
    
    records.append(record)

df_targets = pd.DataFrame(records)

In [7]:
for threshold in [0.05, 0.10, 0.20, 0.30, 0.50]:
    count = len(df_targets[df_targets["overallScore"] >= threshold])
    print(f"Score >= {threshold}: {count} genes")

Score >= 0.05: 2153 genes
Score >= 0.1: 1241 genes
Score >= 0.2: 660 genes
Score >= 0.3: 330 genes
Score >= 0.5: 70 genes


In [8]:
# Filter by overall score
df_filtered = df_targets[df_targets["overallScore"] >= 0.3].copy()
df_filtered = df_filtered.sort_values("overallScore", ascending=False)

print(f"Total before filter: {len(df_targets)}")
print(f"Total after filter (≥0.3): {len(df_filtered)}")
print(f"\nTop 10 genes:")
print(df_filtered[["geneSymbol", "overallScore"]].head(10))
print(f"\nColumns available: {df_filtered.columns.tolist()}")

Total before filter: 7676
Total after filter (≥0.3): 330

Top 10 genes:
  geneSymbol  overallScore
0       TYK2      0.751787
1      IL12B      0.732752
2       IL6R      0.732460
3        TNF      0.716626
4   TRAF3IP2      0.713860
5        MIF      0.697673
6       JAK2      0.693433
7       JAK1      0.690381
8      PADI4      0.689286
9     PTPN22      0.679861

Columns available: ['geneSymbol', 'ensemblId', 'geneName', 'overallScore', 'literature', 'genetic_association', 'clinical', 'animal_model', 'genetic_literature', 'rna_expression']


In [11]:

# Save final version
df_filtered.to_csv("data/raw/opentargets_RA_genes.tsv", sep="\t", index=False)
print("Saved!")

Saved!


In [3]:
import pandas as pd

In [4]:
disease_genes_df = pd.read_csv("data/raw/opentargets_RA_genes.tsv", sep="\t")

In [5]:
disease_genes_df.head()

,geneSymbol,ensemblId,geneName,overallScore,literature,genetic_association,clinical,animal_model,genetic_literature,rna_expression
0,TYK2,ENSG00000105397,tyrosine kinase 2,0.751787,0.781969,0.920610,0.989103,NaN,NaN,NaN
1,IL12B,ENSG00000113302,interleukin 12B,0.732752,0.415165,0.850433,0.983488,NaN,NaN,NaN
2,IL6R,ENSG00000160712,interleukin 6 receptor,0.732460,0.480401,0.803955,0.993177,NaN,NaN,NaN
3,TNF,ENSG00000232810,tumor necrosis factor,0.716626,0.997701,NaN,0.998638,0.480244,0.607931,NaN
4,TRAF3IP2,ENSG00000056972,TRAF3 interacting protein 2,0.713860,0.414003,0.842434,0.950198,0.323343,NaN,0.02473


In [6]:
import networkx as nx

# Load the STRING PPI data
print("Loading PPI network...")
ppi_data = pd.read_csv('data/9606.protein.links.detailed.v12.0.txt', sep=' ')

Loading PPI network...


In [7]:
ppi_data.head()

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000356607,0,0,0,45,134,0,81,173
1,9606.ENSP00000000233,9606.ENSP00000427567,0,0,0,0,128,0,70,154
2,9606.ENSP00000000233,9606.ENSP00000253413,0,0,0,118,49,0,69,151
3,9606.ENSP00000000233,9606.ENSP00000493357,0,0,0,56,53,0,457,471
4,9606.ENSP00000000233,9606.ENSP00000324127,0,0,0,0,46,0,197,201


In [8]:
# Filter for high-confidence interactions
confidence_threshold = 700
ppi_filtered = ppi_data[ppi_data['combined_score'] >= confidence_threshold]
print(f"Before filtering network had {len(ppi_data)} interactions.")
print(f"Filtered network has {len(ppi_filtered)} interactions.")
print(f"Filtering reduced {len(ppi_data) - len(ppi_filtered)} interactions.")

Before filtering network had 13715404 interactions.
Filtered network has 473860 interactions.
Filtering reduced 13241544 interactions.


In [9]:
disease_genes = set(disease_genes_df['geneSymbol'].tolist())
print(f"Loaded {len(disease_genes)} disease-associated genes.")

Loaded 330 disease-associated genes.


In [10]:
# Create the Network Object
G = nx.from_pandas_edgelist(ppi_filtered, 'protein1', 'protein2')
print(f"The final network has {G.number_of_nodes()} proteins and {G.number_of_edges()} interactions.")

The final network has 16201 proteins and 236930 interactions.


In [11]:
# Load the Mapping Info
print("Loading ID mapping info...")
info_df = pd.read_csv('data/9606.protein.info.v12.0.txt', sep='\t')
# Create a dictionary for quick lookup
mapping_dict = dict(zip(info_df['#string_protein_id'], info_df['preferred_name']))

Loading ID mapping info...


In [12]:
ppi_filtered.head()

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
85,9606.ENSP00000000233,9606.ENSP00000158762,0,0,0,47,91,0,814,825
130,9606.ENSP00000000233,9606.ENSP00000357048,0,0,0,79,271,500,260,718
160,9606.ENSP00000000233,9606.ENSP00000262305,0,0,0,0,663,0,866,952
197,9606.ENSP00000000233,9606.ENSP00000329419,0,0,0,89,298,500,316,752
268,9606.ENSP00000000233,9606.ENSP00000469035,0,0,0,335,268,500,259,795


In [11]:
# Translate IDs to Gene Symbols
ppi_filtered['node1'] = ppi_filtered['protein1'].map(mapping_dict)
ppi_filtered['node2'] = ppi_filtered['protein2'].map(mapping_dict)

/tmp/ipykernel_116521/930510982.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ppi_filtered['node1'] = ppi_filtered['protein1'].map(mapping_dict)
/tmp/ipykernel_116521/930510982.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ppi_filtered['node2'] = ppi_filtered['protein2'].map(mapping_dict)


In [12]:
# Drop any rows where mapping failed
ppi_filtered = ppi_filtered.dropna(subset=['node1', 'node2'])

In [13]:
# Create the Network with Gene Symbols
G = nx.from_pandas_edgelist(ppi_filtered, 'node1', 'node2')
print(f"Network built with {G.number_of_nodes()} Gene Symbols.")

Network built with 16201 Gene Symbols.


In [14]:
import pickle
with open('data/processed/network_G.pkl', 'wb') as f:
    pickle.dump(G, f)
print("Network saved!")

Network saved!


In [15]:
my_disease_genes = set(disease_genes_df['geneSymbol'].tolist())

In [17]:
# How many of the disease genes are actually in this network?
overlap = my_disease_genes.intersection(set(G.nodes()))
print(f"Success! {len(overlap)} out of {len(my_disease_genes)} disease genes found in the network.")

Success! 302 out of 330 disease genes found in the network.


In [19]:
# Check which disease genes are missing from the network
missing_genes = my_disease_genes - set(G.nodes())
print(f"Missing genes ({len(missing_genes)}):")
print(sorted(missing_genes))

Missing genes (28):
['ABTB3', 'ANXA3', 'C1orf141', 'C1orf21', 'CFAP20DC', 'CSMD1', 'CYRIB', 'DNASE1L3', 'DRC5', 'GPR174', 'GPR65', 'LBH', 'LONRF2', 'MACIR', 'PCDH20', 'PCYOX1L', 'PLCL2', 'RNF186', 'RTKN2', 'SLC22A25', 'TASL', 'TMEM151B', 'TTC34', 'ZC3H12C', 'ZNF385D', 'ZNF438', 'ZNF774', 'ZNF831']


In [20]:
# To Extract the Disease Module
disease_subgraph = G.subgraph(overlap)

# Find the connected components
components = sorted(nx.connected_components(disease_subgraph), key=len, reverse=True)

if components:
    lcc = components[0] 
    print(f"--- Disease Module Found ---")
    print(f"Total disease genes in network: {len(overlap)}")
    print(f"Genes in the core 'clump' (LCC): {len(lcc)}")
    
    with open('data/disease_module_genes.txt', 'w') as f:
        for gene in lcc:
            f.write(f"{gene}\n")
    print("Core disease module saved to 'data/disease_module_genes.txt'")
else:
    print("No connections found between your disease genes. We might need to lower the STRING threshold.")


--- Disease Module Found ---
Total disease genes in network: 302
Genes in the core 'clump' (LCC): 165
Core disease module saved to 'data/disease_module_genes.txt'


In [21]:
# To get all direct neighbors of disease genes in the FULL network
neighbor_nodes = set()
for gene in overlap:
    neighbors = set(G.neighbors(gene))
    neighbor_nodes.update(neighbors)

In [22]:
# Add neighbors only if they connect multiple disease genes
linker_nodes = set()

for node in neighbor_nodes:
    neighbors_of_neighbor = set(G.neighbors(node))
    disease_connections = neighbors_of_neighbor.intersection(overlap)
    
    # Only add if it bridges 2 or more disease genes
    if len(disease_connections) >= 2:
        linker_nodes.add(node)

# Build refined module
refined_module_nodes = overlap.union(linker_nodes)
refined_subgraph = G.subgraph(refined_module_nodes)

print(f"Disease genes: {len(overlap)}")
print(f"Linker nodes added: {len(linker_nodes)}")
print(f"Refined module size: {len(refined_module_nodes)}")
print(f"Edges: {refined_subgraph.number_of_edges()}")

Disease genes: 302
Linker nodes added: 1994
Refined module size: 2148
Edges: 50777


In [23]:
drugs_df = pd.read_csv('data/interactions.tsv', sep='\t', low_memory=False)
drugs_df = drugs_df.dropna(subset=['gene_name', 'drug_name'])

In [24]:
drug_targets = {}
for _, row in drugs_df.iterrows():
    drug = row['drug_name']
    gene = row['gene_name']
    
    if drug not in drug_targets:
        drug_targets[drug] = set()
    drug_targets[drug].add(gene)

In [25]:
with open('data/drug_targets.pkl', 'wb') as f:
    pickle.dump(drug_targets, f)

In [26]:
# Filter: only to keep drugs whose targets exist in the network G
drug_targets_in_network = {}
for drug, targets in drug_targets.items():
    targets_in_network = targets.intersection(set(G.nodes()))
    if len(targets_in_network) > 0:
        drug_targets_in_network[drug] = targets_in_network

print(f"Drugs with at least one target in network: {len(drug_targets_in_network)}")

Drugs with at least one target in network: 17445
